# ACE-Step singing server for AI Music Composer
Use **Runtime → Change runtime type → T4 GPU** before running. Run the cells in order. This is a temporary college-project demo server: keep this notebook open while generating and stop the runtime when finished.

In [ ]:
!nvidia-smi
!pip -q install uv
!uv python install 3.12
!test -d /content/ACE-Step-1.5 || git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /content/ACE-Step-1.5
%cd /content/ACE-Step-1.5
!uv sync --python 3.12

In [ ]:
import os, secrets, subprocess, time, requests
access_key = secrets.token_urlsafe(24)
server_env = os.environ.copy()
server_env.update({
    'ACESTEP_API_KEY': access_key,
    'ACESTEP_INIT_LLM': 'false',
    'ACESTEP_CONFIG_PATH': 'acestep-v15-turbo',
    'PORT': '8001',
    'HOST': '127.0.0.1'
})
server_log = open('/content/acestep-server.log', 'w')
server_process = subprocess.Popen(
    ['uv', 'run', '--python', '3.12', 'acestep-api'], cwd='/content/ACE-Step-1.5',
    env=server_env, stdout=server_log, stderr=subprocess.STDOUT
)
print('Starting ACE-Step. The first run downloads several GB and can take 10–30 minutes.')
for attempt in range(180):
    if server_process.poll() is not None:
        server_log.flush()
        print(open('/content/acestep-server.log').read()[-5000:])
        raise RuntimeError('ACE-Step stopped during startup.')
    try:
        response = requests.get('http://127.0.0.1:8001/health', headers={'Authorization': f'Bearer {access_key}'}, timeout=5)
        if response.ok:
            print('ACE-Step API is ready.')
            break
    except requests.RequestException:
        pass
    if attempt % 6 == 0:
        print(f'Still starting... {attempt // 6} minute(s)')
    time.sleep(10)
else:
    raise TimeoutError('Startup took longer than 30 minutes. Inspect /content/acestep-server.log')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared.deb
!sudo dpkg -i /content/cloudflared.deb
!cloudflared --version

In [ ]:
import re, shutil, subprocess, time
cloudflared_path = shutil.which('cloudflared')
if not cloudflared_path:
    raise FileNotFoundError('cloudflared is missing. Run the installation cell directly above this one first.')
tunnel_process = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--url', 'http://127.0.0.1:8001', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('The temporary tunnel did not start. Run this cell again.')
print('COPY THESE INTO YOUR LOCAL APP')
print('Colab URL:', public_url)
print('Temporary access key:', access_key)
print('Keep this Colab runtime open while using the app.')

Return to `http://localhost:3000`. Paste **Colab URL** and **Temporary access key** into the orange connection panel, click **Connect Colab**, compose or select a song, and click **Generate realistic vocal**. The URL changes whenever the Colab runtime restarts.